# Imports


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import struct
import sys
import zipfile
from collections import namedtuple
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from dataclasses import dataclass, field
from datetime import datetime
from functools import partial
from multiprocessing import Pool
from pathlib import Path
from typing import List, Optional

import geopandas as gpd
import ggpymanager as ggp
import holoviews as hv
import hvplot.xarray
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydeck as pdk
import pyproj
import rioxarray
import rioxarray as rio
import shapely
import xarray as xr
from compliance_checker.runner import CheckSuite, ComplianceChecker
from dask.diagnostics.progress import ProgressBar
from joblib import Parallel, delayed
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from rasterio.enums import Resampling
from tqdm import tqdm
from windrose import WindroseAxes

import paris_2025 as p
from paris_2025.config import CONFIG
from paris_2025.plotting import RC_PARAMS

Using tracer path: /Users/rmaiwald/Levante/Paris/Input/6_measurements/6_2_tracers


In [3]:
# Get the root logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)

plt.rcParams.update(RC_PARAMS)

In [4]:
def add_midrule_and_bold_last_row(latex_table: str) -> str:
    lines = latex_table.splitlines()

    bottomrule_idx = next(
        (i for i, line in enumerate(lines) if line.strip() == r"\bottomrule"),
        None,
    )
    if bottomrule_idx is None or bottomrule_idx < 1:
        return latex_table

    last_data_idx = bottomrule_idx - 1
    row = lines[last_data_idx].strip()
    if row.endswith(r"\\"):
        row = row[:-2].strip()

    cells = [c.strip() for c in row.split(" & ")]
    lines[last_data_idx] = " & ".join(f"\\textbf{{{c}}}" for c in cells) + r" \\"
    lines.insert(last_data_idx, r"\midrule")

    return "\n".join(lines)

# Fluxes


In [5]:
cadastre_path: str | Path = Path(CONFIG["domain"]["gral"]["conf_path"]) / "cadastre.dat"
source_groups_path: str | Path = p.model_input.fluxes.SOURCE_GROUP_NETCDF_PATH
point_path: str | Path = Path(CONFIG["domain"]["gral"]["conf_path"]) / "point.dat"
cadastre_emissions, source_groups, point_da, GRAL = (
    p.plotting._loaders.load_flux_maps_data(
        cadastre_path, source_groups_path, point_path
    )
)

In [6]:
point_fluxes = point_da.groupby("type").sum()
area_fluxes = cadastre_emissions.sum(["x", "y"]).groupby("type").sum()
location_df = pd.concat(
    [area_fluxes.to_pandas(), point_fluxes.to_pandas()],  # type: ignore
    axis=1,
    keys=["area", "point"],
)
# Convert from kg/h to kt/year
location_df = location_df * 365 * 24 / 1e6
location_df.round(0)

,area,point
type,,
Origins.earth 2023 energie,308.0,751.0
Origins.earth 2023 industrie,582.0,60.0
Origins.earth 2023 residentiel,1894.0,NaN
Origins.earth 2023 respiration_humaine,965.0,NaN
Origins.earth 2023 tertiaire,466.0,NaN
Origins.earth 2023 transport_routier,2707.0,NaN
TNO 2018 Combustion,4696.0,NaN
TNO 2018 Industry,205.0,NaN
TNO 2018 Power,135.0,2481.0


In [7]:
inventories = ["Origins.earth", "TNO"]
totals = pd.DataFrame(
    {inv: location_df[location_df.index.str.contains(inv)].sum() for inv in inventories}
).T
totals["total"] = totals.sum(axis=1)

relative_contributions = totals.div(totals["total"], axis=0) * 100

display(totals.round(0))
display(relative_contributions.round(1))

,area,point,total
Origins.earth,6921.0,811.0,7732.0
TNO,6950.0,2481.0,9431.0


,area,point,total
Origins.earth,89.5,10.5,100.0
TNO,73.7,26.3,100.0


# CO2 Measurements


In [8]:
combined, _ = p.plotting._loaders.load_combined_data()
co2 = ggp.load("co2_measurements", CONFIG).co2.sel(time=slice("2023", "2024")).load()


combined = combined.sortby(
    [~combined.in_gral_domain, "type", "station"],
    ascending=True,
)

co2 = co2.sortby(
    [~co2.in_gral_domain, "type", "station"],
    ascending=True,
)

INFO:root:Opening co2_measurements from /Users/rmaiwald/Levante/Paris/Input/6_measurements/6_2_tracers/co2.nc
INFO:root:Opening concentration_timeseries from /Users/rmaiwald/Levante/Paris/Output/concentration_timeseries.nc


[########################################] | 100% Completed | 7.38 sms


INFO:root:Opening background_co2 from /Users/rmaiwald/Levante/Paris/Output/background_co2.nc
INFO:root:Opening co2_measurements from /Users/rmaiwald/Levante/Paris/Input/6_measurements/6_2_tracers/co2.nc


In [9]:
def create_analysis_table(model, measurements, background):
    """
    Create a table with the following metrics:
    - RMSE
    - Median
    - STD
    - Bias
    - Corr
    - N Observations
    """
    diff = model - measurements
    diff_background = background - measurements
    mean_enhancement = diff_background.mean("time")
    number_of_points = measurements.notnull().sum("time")
    mae = (np.abs(diff)).mean("time")
    rmse = np.sqrt(((diff) ** 2).mean("time"))
    median = diff.median("time")
    corrected_rmse = (diff - diff.mean("station")).std("time")
    corrected_rmse_background = (diff_background - diff_background.mean("station")).std(
        "time"
    )
    mae_background = (np.abs(diff_background)).mean("time")
    rmse_background = np.sqrt(((diff_background) ** 2).mean("time"))
    bias = diff.mean("time")
    corr = xr.corr(measurements, model, dim="time")
    # Create DataFrame
    analysis_table = pd.DataFrame(
        {
            # "mean_enhancement": mean_enhancement.values,
            "RMSE (ppm)": rmse.values,
            "Median (ppm)": median.values,
            "STD (ppm)": diff.std("time").values,
            # "rmse_background": rmse_background.values,
            # "MAE (ppm)": mae.values,
            # "mae_background": mae_background.values,
            # "corrected_rmse": corrected_rmse.values,
            # "corrected_rmse_background": corrected_rmse_background.values,
            "Bias (ppm)": bias.values,
            "Corr": corr.values,
            "N Observations": number_of_points.values,
        },
        index=mean_enhancement.station.values,
    )
    # Add a new line with the mean of each column
    analysis_table.loc["Mean"] = analysis_table.mean()
    analysis_table = analysis_table.round(2)
    analysis_table["N Observations"] = analysis_table["N Observations"].astype(int)
    return analysis_table

In [10]:
for_table = combined.sel(station=~combined.station.str.contains("Mean"))  # Drop Mean

dfs = []
for hours in [range(0, 24), range(12, 17)]:
    filtered = for_table.where(for_table.time.dt.hour.isin(hours), drop=True)
    dfs.append(
        create_analysis_table(
            # filtered.sel(dataset="TNO"),
            filtered.sel(dataset="Origins.earth"),
            filtered.sel(dataset="CO2"),
            filtered.sel(dataset="Background"),
        )
    )
# Create one dataframe with multiindex in columns
analysis_df = pd.concat(
    dfs,
    axis=1,
    keys=("Full day (00-24 UTC)", "Afternoon (12-17 UTC)"),
)
# Convert to a DataFrame and select relevant columns
location_df = (
    co2.coords.to_dataset()
    .drop_vars("time")
    .to_pandas()[
        [
            "height",
            "latitude",
            "longitude",
            "instrument",
            "in_gral_domain",
        ]
    ]
)

# Round height to the nearest meter
location_df["height"] = location_df["height"].round().astype(int)

# Rename columns
location_df = location_df.rename(
    columns={
        "station": "Station",
        "height": "Height (m a.g.l.)",
        "latitude": "Latitude",
        "longitude": "Longitude",
        "instrument": "Instrument",
        "in_gral_domain": "In GRAL Domain",
    }
)

# Round latitude and longitude to 4 decimal places
location_df["Latitude"] = location_df["Latitude"].round(3)
location_df["Longitude"] = location_df["Longitude"].round(3)

In [11]:
# Clean up strings
for df in [location_df, analysis_df]:
    # Escape underscores in the index for LaTeX
    df.index = df.index.str.replace("_", "\\_", regex=False)

    # Remove all True values except the first one and replace with "Yes"
    if "In GRAL Domain" in df.columns:
        df["In GRAL Domain"] = df["In GRAL Domain"].replace({True: "Yes", False: "No"})
        contains_yes = df.index[df["In GRAL Domain"] == "Yes"]
        contains_no = df.index[df["In GRAL Domain"] == "No"]
        df.loc[contains_yes[1:], "In GRAL Domain"] = ""
        df.loc[contains_no[1:], "In GRAL Domain"] = ""

In [12]:
station_averaged_afternoon_diff = (
    combined.sel(dataset=["CO2", "Origins.earth"])
    .sel(time=combined.time.dt.hour.isin(range(11, 17)))
    .resample(time="1D")
    .mean()
    .mean("station")
    .diff("dataset")
    .squeeze()
)
display(np.sqrt((station_averaged_afternoon_diff**2).mean("time"))), display(
    station_averaged_afternoon_diff.mean()
)

<xarray.DataArray ()> Size: 8B
array(3.52620104)
Coordinates:
    loss_type  <U27 108B 'rmse - filter: True'
    prior      <U13 52B 'Origins.earth'
    dataset    <U13 52B 'Origins.earth'
Attributes:
    units_metadata:          temperature: on_scale
    long_name:               Height-binned background CO2 (station-matched)
    units:                   ppm
    height_bin_edges_m_agl:  [0, 40, 80, 120, 200]

<xarray.DataArray ()> Size: 8B
array(-0.84271622)
Coordinates:
    loss_type  <U27 108B 'rmse - filter: True'
    prior      <U13 52B 'Origins.earth'
    dataset    <U13 52B 'Origins.earth'
Attributes:
    units_metadata:          temperature: on_scale
    long_name:               Height-binned background CO2 (station-matched)
    units:                   ppm
    height_bin_edges_m_agl:  [0, 40, 80, 120, 200]

(None, None)

In [13]:
display(analysis_df)
display(location_df)

Full day (00-24 UTC)                                          \
                       RMSE (ppm) Median (ppm) STD (ppm) Bias (ppm)  Corr   
CDS\_34                     13.28         2.06     13.04       2.55  0.77   
JUS\_30                     11.95        -0.17     11.95      -0.19  0.79   
JUS\_40                     11.13        -0.05     11.13       0.05  0.76   
MEU\_45                      5.12        -0.91      4.78      -1.83  0.92   
MEU\_65                      4.41        -0.65      4.17      -1.45  0.93   
MEU\_90                      3.34        -0.38      3.22      -0.85  0.96   
ROV\_103                     5.35        -0.28      5.27      -0.93  0.90   
BAS\_50                     10.08        -1.23     10.03      -0.99  0.70   
BED\_32                     17.07        -2.16     16.15      -5.54  0.72   
BNF\_80                     11.25        -1.21     10.96      -2.52  0.70   
BOB\_54                     15.55        -5.42     12.93      -8.63  0.72   
CAP\_30                     12.81        -3.27     12.70      -1.72  0.59   
CB2\_165                    10.76        -3.49      9.17      -5.63  0.78   
CDS\_34\_K96                10.85        -0.78     10.76      -1.37  0.77   
CIT\_88                      9.57        -0.16      9.42      -1.70  0.78   
EIF\_34                      9.97         0.21      9.88       1.28  0.81   
GUS\_65                     12.63        -5.45     10.42      -7.14  0.82   
HBO\_80                     13.47        -1.87     12.69      -4.51  0.71   
JUS\_33                     11.21        -0.12     11.12       1.40  0.79   
MON\_16                     10.85        -3.75      9.59      -5.09  0.82   
NAS\_110                    12.56        -7.26      9.21      -8.54  0.78   
NEY\_55                     12.69        -1.76     12.10      -3.81  0.75   
OBS\_27                     10.15         0.08     10.15       0.06  0.83   
PLA\_51                     10.69        -1.51     10.37      -2.57  0.77   
POM\_45                     11.08        -1.59     11.01      -1.22  0.78   
TF1\_60                     13.13        -3.76     11.50      -6.34  0.68   
Mean                        10.81        -1.73     10.14      -2.58  0.78   

                            Afternoon (12-17 UTC)                         \
             N Observations            RMSE (ppm) Median (ppm) STD (ppm)   
CDS\_34               16272                  9.45         2.28      8.63   
JUS\_30               14121                  7.53        -1.17      7.42   
JUS\_40                9758                  7.03         0.19      6.95   
MEU\_45               11088                  3.18        -0.70      3.02   
MEU\_65               11101                  3.13        -0.74      2.95   
MEU\_90               16491                  2.54        -0.43      2.48   
ROV\_103              15623                  4.65        -0.11      4.63   
BAS\_50                3929                  6.77        -1.43      6.75   
BED\_32                6936                  7.29        -1.56      6.96   
BNF\_80                8872                  5.64        -0.65      5.64   
BOB\_54                9872                  6.10        -3.44      4.74   
CAP\_30                1138                 10.58        -3.95     10.33   
CB2\_165               9313                 10.79        -4.77      8.67   
CDS\_34\_K96           6461                  5.24        -0.42      5.24   
CIT\_88                8216                  6.17         0.30      6.07   
EIF\_34                8065                  8.76         0.17      8.54   
GUS\_65                5543                  9.21        -4.08      7.56   
HBO\_80                8976                  7.90        -0.82      7.89   
JUS\_33                7282                  6.75        -0.92      6.74   
MON\_16                7828                  6.45        -3.33      5.33   
NAS\_110               8143                  7.54        -5.21      5.56   
NEY\_55                7126    

,Height (m a.g.l.),Latitude,Longitude,Instrument,In GRAL Domain
station,,,,,
CDS\_34,34,48.896,2.388,Picarro,Yes
JUS\_30,30,48.846,2.356,Picarro,
JUS\_40,40,48.846,2.356,Picarro,
MEU\_45,45,48.802,2.204,Picarro,
MEU\_65,65,48.802,2.204,Picarro,
MEU\_90,90,48.802,2.204,Picarro,
ROV\_103,103,48.885,2.422,Picarro,
BAS\_50,50,48.852,2.370,K96,
BED\_32,32,48.820,2.371,HPP,


In [14]:
# Convert to LaTeX
analysis_latex = analysis_df.to_latex(
    index=True,
    multirow=True,
    multicolumn=True,
    float_format="%.2f",
)
analysis_latex = add_midrule_and_bold_last_row(analysis_latex)
print(analysis_latex)

\begin{tabular}{lrrrrrrrrrrrr}
\toprule
 & \multicolumn{6}{r}{Full day (00-24 UTC)} & \multicolumn{6}{r}{Afternoon (12-17 UTC)} \\
 & RMSE (ppm) & Median (ppm) & STD (ppm) & Bias (ppm) & Corr & N Observations & RMSE (ppm) & Median (ppm) & STD (ppm) & Bias (ppm) & Corr & N Observations \\
\midrule
CDS\_34 & 13.28 & 2.06 & 13.04 & 2.55 & 0.77 & 16272 & 9.45 & 2.28 & 8.63 & 3.86 & 0.86 & 3396 \\
JUS\_30 & 11.95 & -0.17 & 11.95 & -0.19 & 0.79 & 14121 & 7.53 & -1.17 & 7.42 & -1.28 & 0.87 & 2945 \\
JUS\_40 & 11.13 & -0.05 & 11.13 & 0.05 & 0.76 & 9758 & 7.03 & 0.19 & 6.95 & 1.07 & 0.88 & 2031 \\
MEU\_45 & 5.12 & -0.91 & 4.78 & -1.83 & 0.92 & 11088 & 3.18 & -0.70 & 3.02 & -1.00 & 0.97 & 2237 \\
MEU\_65 & 4.41 & -0.65 & 4.17 & -1.45 & 0.93 & 11101 & 3.13 & -0.74 & 2.95 & -1.05 & 0.97 & 2246 \\
MEU\_90 & 3.34 & -0.38 & 3.22 & -0.85 & 0.96 & 16491 & 2.54 & -0.43 & 2.48 & -0.53 & 0.97 & 3444 \\
ROV\_103 & 5.35 & -0.28 & 5.27 & -0.93 & 0.90 & 15623 & 4.65 & -0.11 & 4.63 & -0.44 & 0.92 & 3257 \\
BAS

In [15]:
location_latex = location_df.to_latex(
    index=True,
    float_format="%.3f",
)
print(location_latex)

\begin{tabular}{lrrrll}
\toprule
 & Height (m a.g.l.) & Latitude & Longitude & Instrument & In GRAL Domain \\
station &  &  &  &  &  \\
\midrule
CDS\_34 & 34 & 48.896 & 2.388 & Picarro & Yes \\
JUS\_30 & 30 & 48.846 & 2.356 & Picarro &  \\
JUS\_40 & 40 & 48.846 & 2.356 & Picarro &  \\
MEU\_45 & 45 & 48.802 & 2.204 & Picarro &  \\
MEU\_65 & 65 & 48.802 & 2.204 & Picarro &  \\
MEU\_90 & 90 & 48.802 & 2.204 & Picarro &  \\
ROV\_103 & 103 & 48.885 & 2.422 & Picarro &  \\
BAS\_50 & 50 & 48.852 & 2.370 & K96 &  \\
BED\_32 & 32 & 48.820 & 2.371 & HPP &  \\
BNF\_80 & 80 & 48.833 & 2.377 & K96 &  \\
BOB\_54 & 54 & 48.908 & 2.444 & K96 &  \\
CAP\_30 & 30 & 48.863 & 2.291 & HPP &  \\
CB2\_165 & 165 & 48.889 & 2.251 & HPP &  \\
CDS\_34\_K96 & 34 & 48.895 & 2.387 & K96 &  \\
CIT\_88 & 88 & 48.828 & 2.231 & HPP &  \\
EIF\_34 & 34 & 48.855 & 2.293 & K96 &  \\
GUS\_65 & 65 & 48.794 & 2.348 & HPP &  \\
HBO\_80 & 80 & 48.908 & 2.310 & K96 &  \\
JUS\_33 & 33 & 48.846 & 2.356 & K96 &  \\
MON\_16 & 16 & 48

# Ensemble size and spread


In [16]:
conc_series = ggp.load("concentration_timeseries", CONFIG)
loss_diff = conc_series["loss_diff"].sel(loss_type="rmse - filter: True")
mask = (loss_diff < 0.1).sum("best_sim_id")
print(
    "Take the 90% percentile as a threshold for the number of simulations that are "
    "below the 0.1 RMSE threshold."
)
display(mask.to_pandas().describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
limit = int((0.9 * len(mask)))
print(f"Select top 10% of the simulations: {len(mask) - limit}/{len(mask)}")
mean = mask.sortby(mask).values[limit:].mean()
print(
    f"Mean number of simulations below the 0.1 RMSE threshold for the top 10%: "
    f"{mean:.2f}"
)
print(
    f"Percent of times with exactly 0 simulations below the 0.1 RMSE threshold: "
    f"{(mask == 1).mean().values * 100:.2f}%"
)

INFO:root:Opening concentration_timeseries from /Users/rmaiwald/Levante/Paris/Output/concentration_timeseries.nc


Take the 90% percentile as a threshold for the number of simulations that are below the 0.1 RMSE threshold.


count    17544.000000
mean         3.912677
std          4.341492
min          1.000000
25%          2.000000
50%          3.000000
75%          5.000000
90%          7.000000
95%         10.000000
99%         24.000000
max         85.000000
Name: loss_diff, dtype: float64

Select top 10% of the simulations: 1755/17544
Mean number of simulations below the 0.1 RMSE threshold for the top 10%: 13.45
Percent of times with exactly 0 simulations below the 0.1 RMSE threshold: 18.35%


In [17]:
co2_enhancement = (
    conc_series.co2_timeseries.sel(loss_type="rmse - filter: True")
    .sel(type=conc_series.type.str.contains("Origins.earth|VPRM"))
    .sum("type")
    .where(loss_diff < 0.1)
    .compute()
)
ensemble_spread = (
    co2_enhancement.max("best_sim_id") - co2_enhancement.min("best_sim_id")
).mean("station")

In [18]:
display(
    ensemble_spread.to_pandas().describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
)
# Largest 10%
mean = ensemble_spread.sel(time=mask.time.sortby(mask).values[limit:]).mean()
print(
    f"Mean ensemble spread for the top 10% of simulations with the lowest RMSE: "
    f"{mean:.2f} ppm"
)

count    17544.000000
mean         5.442394
std         11.120522
min          0.000000
25%          0.844529
50%          2.136604
75%          5.091579
90%         12.631622
95%         21.996115
99%         57.793079
max        137.973740
Name: co2_timeseries, dtype: float64

Mean ensemble spread for the top 10% of simulations with the lowest RMSE: 26.95 ppm


# Wind Measurements


In [19]:
model_meteo_timeseries = (
    ggp.load("model_meteo_timeseries", CONFIG)
    .sel(loss_type="rmse - filter: True")
    .load()
)
meteo = ggp.load("meteo_measurements", CONFIG)
meteo = meteo.sel(
    station=model_meteo_timeseries.station, time=model_meteo_timeseries.time
).load()

INFO:root:Opening gramm_meteo_timeseries from /Users/rmaiwald/Levante/Paris/Output/gramm_meteo_timeseries.nc
INFO:root:Opening gral_meteo_timeseries from /Users/rmaiwald/Levante/Paris/Output/gral_meteo_timeseries.nc
INFO:root:Selecting model data for station LONGCHAMP from model gral
INFO:root:Selecting model data for station PARIS-MONTSOURIS from model gral
INFO:root:Selecting model data for station TOUR EIFFEL from model gramm
INFO:root:Selecting model data for station LFPB from model gramm
INFO:root:Selecting model data for station LFPN_2 from model gramm
INFO:root:Selecting model data for station LFPO from model gramm
INFO:root:Selecting model data for station LFPV from model gramm
INFO:root:Selecting model data for station Cité des Sciences from model gral
INFO:root:Selecting model data for station Meudon from model gramm
INFO:root:Selecting model data for station Romainville from model gral
INFO:root:Opening meteo_measurements from /Users/rmaiwald/Levante/Paris/Input/6_measuremen

In [20]:
wind_speed_diff = model_meteo_timeseries["wind_speed"] - meteo["wind_speed"]
wind_direction_diff = ggp.processing.circular_diff(
    model_meteo_timeseries["wind_direction"], meteo["wind_direction"]
)
u_wind_diff = model_meteo_timeseries["u"] - meteo["u_wind"]
v_wind_diff = model_meteo_timeseries["v"] - meteo["v_wind"]

In [21]:
def model_performance_df(
    model: xr.Dataset,
    meteo: xr.Dataset,
) -> pd.DataFrame:
    """Create a DataFrame of model performance metrics from wind model and measurements.

    Computes bias, MAE, RMSE, and Pearson R per station averaged over time and
    (if present) all ensemble members (``best_sim_id`` dimension). u and v are
    combined into a single wind vector variable: bias is reported as separate u/v
    columns, MAE and RMSE are computed on the vector magnitude (no R for the vector).
    An additional column reports the mean observed wind speed per station.

    Parameters
    ----------
    model : xr.Dataset
        Model output with variables ``wind_speed``, ``wind_direction``, ``u``, ``v``.
    meteo : xr.Dataset
        Measurements with variables ``wind_speed``, ``wind_direction``,
        ``u_wind``, ``v_wind``.

    Returns
    -------
    pd.DataFrame
        Multi-level column DataFrame indexed by station with columns
        ``(variable, metric)``, plus a final "Mean" row.
    """

    def _avg_dims(da: xr.DataArray) -> list[str]:
        return [d for d in da.dims if d != "station"]

    def _metrics(model_var: xr.DataArray, meas_var: xr.DataArray) -> pd.DataFrame:
        diff: xr.DataArray = (model_var - meas_var).astype(float)  # type: ignore[assignment]
        avg_dims = _avg_dims(diff)
        bias = diff.mean(dim=avg_dims)
        mae = abs(diff).mean(dim=avg_dims)
        rmse = (diff**2).mean(dim=avg_dims) ** 0.5
        r = xr.corr(model_var.astype(float), meas_var.astype(float), dim=avg_dims)  # type: ignore[arg-type]
        return pd.DataFrame(
            {
                "Bias (m/s)": bias.to_pandas(),
                # "MAE": mae.to_pandas(),
                "RMSE (m/s)": rmse.to_pandas(),
                "R": r.to_pandas(),
            }
        )

    def _vector_metrics(
        u_model: xr.DataArray,
        v_model: xr.DataArray,
        u_meas: xr.DataArray,
        v_meas: xr.DataArray,
    ) -> pd.DataFrame:
        fu: xr.DataArray = (u_model - u_meas).astype(float)  # type: ignore[assignment]
        fv: xr.DataArray = (v_model - v_meas).astype(float)  # type: ignore[assignment]
        avg_dims = _avg_dims(fu)
        u_bias = fu.mean(dim=avg_dims)
        v_bias = fv.mean(dim=avg_dims)
        mae = ((fu**2 + fv**2) ** 0.5).mean(dim=avg_dims)
        rmse = ((fu**2 + fv**2).mean(dim=avg_dims)) ** 0.5
        return pd.DataFrame(
            {
                # "Bias (u)": u_bias.to_pandas(),
                # "Bias (v)": v_bias.to_pandas(),
                # "MAE": mae.to_pandas(),
                "RMSE (m/s)": rmse.to_pandas(),
            }
        )

    wind_direction_diff = ggp.processing.circular_diff(
        model["wind_direction"], meteo["wind_direction"]
    )
    wind_dir_corr = xr.corr(
        model["wind_direction"].astype(float),  # type: ignore[arg-type]
        meteo["wind_direction"].astype(float),  # type: ignore[arg-type]
        dim=_avg_dims(wind_direction_diff),
    )

    obs_speed = (
        meteo["wind_speed"]
        .astype(float)
        .mean(dim=_avg_dims(meteo["wind_speed"]))  # type: ignore[assignment]
    )

    frames: dict[str, pd.DataFrame] = {
        "Altitude (m a.s.l.)": pd.DataFrame({"(m)": meteo["altitude"].to_pandas()}),
        "Obs. speed": pd.DataFrame({"(m/s)": obs_speed.to_pandas()}),
        "Vector": _vector_metrics(
            model["u"], model["v"], meteo["u_wind"], meteo["v_wind"]
        ),
        "Speed": _metrics(model["wind_speed"], meteo["wind_speed"]),
        "Dir. (°)": pd.DataFrame(
            {
                # "Bias": wind_direction_diff.astype(float).mean(dim=_avg_dims(wind_direction_diff)).to_pandas(),  # type: ignore[assignment]
                # "MAE": abs(wind_direction_diff.astype(float)).mean(dim=_avg_dims(wind_direction_diff)).to_pandas(),  # type: ignore[assignment]
                "RMSE": ((wind_direction_diff.astype(float) ** 2).mean(dim=_avg_dims(wind_direction_diff)) ** 0.5).to_pandas(),  # type: ignore[assignment]
                # "R": wind_dir_corr.to_pandas(),
            }
        ),
    }

    df = pd.concat(frames, axis=1)

    mean_row = df.mean().rename("Mean").to_frame().T
    df = pd.concat([df, mean_row])
    df.index.name = "Station"
    return df


location_df = model_performance_df(model_meteo_timeseries, meteo).round(2)

# Replace all underscores in the index with escaped underscores for LaTeX
location_df.index = location_df.index.str.replace("_", "\\_", regex=False)

In [22]:
latex = location_df.to_latex(float_format="%.2f")

latex = add_midrule_and_bold_last_row(latex)
print(latex)
display(location_df.style.background_gradient(axis="index").format("{:.2f}"))

\begin{tabular}{lrrrrrrr}
\toprule
 & Altitude (m a.s.l.) & Obs. speed & Vector & \multicolumn{3}{r}{Speed} & Dir. (°) \\
 & (m) & (m/s) & RMSE (m/s) & Bias (m/s) & RMSE (m/s) & R & RMSE \\
Station &  &  &  &  &  &  &  \\
\midrule
LONGCHAMP & 27.00 & 2.51 & 1.39 & 0.25 & 1.13 & 0.75 & 46.44 \\
PARIS-MONTSOURIS & 75.00 & 3.11 & 1.63 & -0.97 & 1.37 & 0.74 & 29.57 \\
TOUR EIFFEL & 330.00 & 7.20 & 2.65 & -0.56 & 2.36 & 0.80 & 19.57 \\
LFPB & 65.00 & 3.59 & 2.20 & -0.70 & 1.45 & 0.78 & 49.23 \\
LFPN\_2 & 164.00 & 3.83 & 1.81 & -0.66 & 1.40 & 0.83 & 40.45 \\
LFPO & 96.00 & 3.73 & 1.78 & -0.49 & 1.30 & 0.80 & 37.96 \\
LFPV & 179.00 & 3.71 & 1.79 & -1.00 & 1.46 & 0.81 & 34.67 \\
Cité des Sciences & 43.00 & 2.64 & 1.60 & 0.63 & 1.16 & 0.80 & 29.56 \\
Meudon & 173.00 & 4.72 & 3.34 & -0.92 & 1.79 & 0.74 & 41.01 \\
Romainville & 128.00 & 6.03 & 3.22 & -0.64 & 2.02 & 0.75 & 27.87 \\
\midrule
\textbf{Mean} & \textbf{128.00} & \textbf{4.11} & \textbf{2.14} & \textbf{-0.51} & \textbf{1.54} & \textbf{0